# Build scenario datasets

**Summary**

This notebook builds the CO2 emissions datasets used to drive CMIP simulations from the early 1990s (IS92 CMIP1) through to the latest scenarios in CMIP7 (2026).

**Authors**

Paul J. Durack ([durack1](https://github.com/durack1));

**Reference**

None

**References**

Pederson *et al.* (2021) doi: [10.1016/j.gloenvcha.2020.102199](https://doi.org/10.1016/j.gloenvcha.2020.102199)

**Notes**

PJD 19 Apr 2026 - Initial start<br>
PJD 20 Apr 2026 - Add 5 SA90<br>
PJD 21 Apr 2026 - Add 40 SRES; Add 4 RCPs<br>
PJD 22 Apr 2026 - Add 127 SSPs<br>
PJD 23 Apr 2026 - Update data ref/links; replace `null` with `None`<br>

**Notebook lookup table**

Note: Links to the sections work best when viewing this notebook via [nbviewer](https://nbviewer.org/github/durack1/CMIPSummary/blob/main/figuresAndTables.ipynb).

1. [Figure 1: <SOMETHING>](buildData.ipynb#fig1)


### imports

In [ ]:
%%time
import matplotlib.pyplot as plt
import numpy as np
import copy
import datetime
import json
import os

## SA90

**time periods:** 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025, 2050, 2075, 2100<br>
**scenarios:** A, B, C, D, E<br>
**reference:** Tirpak and Vellinga, 1990: ["Emissions Scenarios (Chapter 2)"](https://www.ipcc.ch/site/assets/uploads/2018/03/ipcc_far_wg_III_chapter_02.pdf#page=6) In: Bernthal *et al.*, 1990: [FAR Climate Change: The IPCC Response Strategies](https://www.ipcc.ch/report/ar1/wg3/)

In [ ]:
%%time
data = {
    "time": {"values": [1985, 2000, 2025, 2050, 2075, 2100], "unit": "year"},
    "variable": "carbon_dioxide_emissions",
    "unit": "Petagrams C/Yr",
    "scenarios": {
        "2030 High Emissions (Average) Scenario": [6.0, 7.7, 11.5, 15.2, 18.7, 22.4],
        "2060 Low Emissions (Average) Scenario": [5.9, 5.5, 6.4, 7.5, 8.8, 10.3],
        "Control Policies (Average) Scenario": [5.9, 5.6, 6.3, 7.1, 5.1, 3.5],
        "Accelerated Policies (Average) Scenario": [6.0, 5.6, 5.1, 2.9, 3.0, 2.7],
        "Alternative Accelerated Policies Scenario": [6.0, 4.6, 3.8, 3.7, 3.5, 2.6],
    },
}

## IS92

**time periods:** 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025, 2050, 2075, 2100<br>
**scenarios:** A, B, C, D, E, F<br>
**references:**<br>
Leggett *et al.*, 1992: ["Emissions Scenarios for the IPCC:
an Update (Section A3)"](https://www.ipcc.ch/site/assets/uploads/2018/05/ipcc_wg_I_1992_suppl_report_section_a3.pdf) In: Houghton *et al.*, 1992: [Climate Change 1992: The Supplementary Report to the IPCC Scientific Assessment](https://www.ipcc.ch/report/climate-change-1992-the-supplementary-report-to-the-ipcc-scientific-assessment/)<br>
Mitchell and Gregory, 1992: ["ANNEX: Climatic consequences of emissions and a comparison of IS92a and SA90"](https://www.ipcc.ch/site/assets/uploads/2018/05/ipcc_wg_I_1992_suppl_report_annex.pdf) In: Houghton *et al.*, 1992: [Climate Change 1992: The Supplementary Report to the IPCC Scientific Assessment](https://www.ipcc.ch/report/climate-change-1992-the-supplementary-report-to-the-ipcc-scientific-assessment/)<br>
Pepper *et al.*, 1992 (May): ["Emissions Scenarios for the IPCC An Update: Assumptions, Methodology, and Results"](https://www.researchgate.net/publication/338901778_Emission_Scenarios_for_the_IPCC_-_An_Update_May_1992)

In [ ]:
%%time
data = {
    "time": {"values": [1990, 2000, 2025, 2050, 2100], "unit": "year"},
    "variable": "carbon_dioxide_emissions",
    "unit": "Bt C",
    "scenarios": {
        "IS92a updated": [7.4, 8.4, 12.2, 14.5, 20.3], # updated - GtC Leggett - June 1990
        "IS92a": [6.0, 7.0, 10.7, 13.2, 19.8], # Pepper p45/table 3.1.16 p39
        "IS92b": [6.0, 6.8, 10.3, 12.5, 18.6], # Pepper p46/table 3.1.17 p40
        "IS92c": [6.0, 6.0, 7.4, 6.5, 4.6], # Pepper p47/table 3.1.18 p41
        "IS92d": [6.0, 6.4, 8.6, 8.4, 9.9], # Pepper p48/table 3.1.19 p42
        "IS92e": [6.0, 7.6, 13.5, 18.6, 34.9], # Pepper p49/table 3.1.20 p43
        "IS92f": [6.0, 7.3, 12.6, 15.8, 25.9], # Pepper p50/table 3.1.21 p44
    },
}

## SRES

**time periods:** 1990, 2000, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100<br>
**scenarios:** A1, A1C, A1G, A1v1, A1v2, A1T, A2, A2G, A2-A1, B1, B1T, B1High, B2, B2High<br>
**references:**<br>
Nakicenovic and Swart, 2000: [Emission Scenarios](https://www.ipcc.ch/report/emissions-scenarios/)<br>
[VII Statistical tables - Table VII.1](https://www.ipcc.ch/site/assets/uploads/2018/03/emissions_scenarios-1.pdf#page=387) (pp 381-576)


In [ ]:
%%time
data = {
    "time": {"values": [1990, 2000, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100], "unit": "year"},
    "variable": "anthropogenic carbon dioxide emissions (World total CO2)",
    "unit": "GtC",
    "scenarios": {
        "A1B-AIM": [7.10, 7.97, 10.88, 12.64, 14.48, 15.35, 16.38, 16.00, 15.73, 15.18, 14.30, 13.49], # Marker scenario
        "A1B-ASF": [7.10, 7.97, 11.13, 15.91, 20.62, 23.59, 26.56, 24.72, 22.87, 21.15, 19.53, 17.92],
        "A1B-IMAGE": [7.10, 7.97, 9.27, 12.11, 15.00, 17.61, 18.71, 18.45, 18.02, 16.96, 15.07, 13.39],
        "A1B-MARIA": [7.10, 7.97, 8.80, 9.61, 10.44, 11.51, 13.01, 12,87, 13.50, 14.07, 15.86, 16.44],
        "A1B-MESSAGE": [7.10, 7.97, 9.36, 10.81, 13.33, 15.56, 16.45, 17.81, 18.70, 17.83, 15.44, 13.83],
        "A1B-MiniCAM": [7.10, 7.97, 9.60, 12.33, 15.13, 17.35, 19.19, 19.64, 20.24, 20.98, 18.43, 15.96],
        "A1C-AIM": [7.10, 7.97, 11.79, 16.12, 20.06, 22.88, 26.91, 28.51, 30.26, 32.23, 34.41, 36.75],
        "A1C-MESSAGE": [7.10, 7.97, 9.65, 11.23, 14.17, 17.16, 20.62, 24.47, 27.33, 29.03, 30.78, 32.07],
        "A1C-MiniCAM": [7.10, 7.97, 9.66, 12.54, 16.14, 20.48, 25.37, 26.27, 26.64, 26.46, 26.09, 25.89],
        "A1G-AIM": [7.10, 7.97, 11.11, 14.87, 18.01, 21.44, 25.62, 27.08, 28.67, 29.76, 30.26, 30.79],
        "A1G-MESSAGE": [7.10, 7.97, 9.54, 10.91, 14.12, 17.61, 21.42, 26.08, 29.04, 30.55, 31.13, 30.31],
        "A1FI-MiniCAM": [7.10, 7.97, 9.73, 12.73, 16.19, 19.97, 23.90, 25.69, 27.28, 28.68, 28.42, 28.24], # Previously A1G-MiniCAM
        "A1T-AIM": [7.10, 7.97, 9.81, 11.52, 12.15, 11.67, 11.28, 10.49, 9.63, 9.29, 8.97, 8.66],
        "A1T-MESSAGE": [7.10, 7.97, 9.38, 10.26, 12.38, 12.65, 12.26, 11.38, 9.87, 8.02, 6.26, 4.32],
        "A1T-MARIA": [7.10, 7.97, 8.70, 9.39, 9.82, 10.58, 11.26, 10.43, 9.85, 9.18, 8.94, 9.10],
        "A1v1-MiniCAM": [7.10, 7.97, 9.16, 11.35, 13.29, 14.97, 16.53, 17.14, 17.53, 17.70, 16.83, 15.98],
        "A1v2-MiniCAM": [7.10, 7.97, 9.23, 11.42, 13.34, 14.78, 16.10, 16.49, 16.80, 17.04, 16.47, 15.93],
        "A2-AIM": [7.10, 7.97, 10.22, 11.36, 13.72, 15.31, 17.23, 19.47, 22.07, 25.38, 29.56, 34.47],
        "A2-ASF": [7.10, 7.97, 9.58, 12.25, 14.72, 16.07, 17.43, 19.16, 20.89, 23.22, 26.15, 29.09],
        "A2G-IMAGE": [7.10, 7.97, 10.02, 12.08, 14.39, 16.70, 19.01, 19.12, 19.23, 19.34, 19.45, 19.56],
        "A2-MESSAGE": [7.10, 7.97, 9.45, 11.46, 13.33, 14.62, 15.91, 17.05, 18.76, 21.32, 25.00, 28.19],
        "A2-MiniCAM": [7.10, 7.97, 9.12, 10.83, 12.19, 13.96, 16.15, 18.44, 20.83, 23.32, 26.19, 29.30],
        "A2-A1-MiniCAM": [7.10, 7.97, 8.39, 8.51, 8.50, 9.24, 11.05, 12.52, 14.87, 18.08, 20.38, 22.89],
        "B1-AIM": [7.10, 7.97, 9.39, 10.28, 11.22, 11.69, 12.35, 10.84, 9.67, 8.20, 7.06, 6.15],
        "B1-ASF": [7.10, 7.97, 10.76, 14.46, 16.85, 17.59, 18.33, 15.29, 12.24, 9.86, 8.13, 6.41],
        "B1-IMAGE": [7.10, 7.97, 9.28, 10.63, 11.11, 11.72, 11.29, 9.74, 8.18, 6.70, 5.32, 4.23],
        "B1-MARIA": [7.10, 7.97, 8.25, 8.71, 9.31, 9.85, 9.64, 8.80, 8.05, 7.88, 7.90, 8.19],
        "B1-MESSAGE": [7.10, 7.97, 9.05, 9.16, 9.27, 9.25, 8.57, 7.74, 6.75, 5.79, 4.63, 4.04],
        "B1-MiniCAM": [7.10, 7.97, 8.38, 9.53, 10.13, 9.94, 9.45, 8.94, 8.08, 6.87, 6.07, 5.28],
        "B1T-MESSAGE": [7.10, 7.97, 9.05, 9.08, 9.11, 8.95, 7.81, 6.63, 5.35, 4.35, 3.35, 2.68],
        "B1High-MESSAGE": [7.10, 7.97, 8.69, 8.96, 9.32, 9.61, 9.44, 8.94, 8.03, 7.06, 6.38, 5.68],
        "B1High-MiniCAM": [7.10, 7.97, 8.82, 10.48, 11.70, 12.09, 12.18, 12.35, 12.21, 11.74, 11.08, 10.43],
        "B2-AIM": [7.10, 7.97, 9.43, 10.47, 11.90, 13.36, 15.20, 15.07, 15.01, 14.76, 14.41, 14.04],
        "B2-ASF": [7.10, 7.97, 9.97, 12.73, 14.75, 15.51, 16.27, 16.52, 16.78, 17.35, 18.24, 19.13],
        "B2-IMAGE": [7.10, 7.97, 9.19, 10.40, 11.06, 11.71, 12.36, 12.04, 11.72, 11.40, 11.07, 10.75],
        "B2-MARIA": [7.10, 7.97, 8.84, 9.93, 11.05, 12.70, 13.70, 14.55, 14.96, 15.16, 15.20, 15.15],
        "B2-MESSAGE": [7.10, 7.97, 8.78, 9.05, 9.90, 10.69, 11.01, 11.49, 11.62, 12.15, 12.79, 13.32],
        "B2-MiniCAM": [7.10, 7.97, 8.88, 10.48, 11.51, 12.27, 13.18, 13.85, 14.34, 14.67, 14.73, 14.82],
        "B2C-MARIA": [7.10, 7.97, 9.12, 10.67, 12.85, 14.76, 15.51, 16.81, 17.74, 18.57, 19.16, 19.71],
        "B2High-MiniCAM": [7.10, 7.97, 9.23, 11.30, 13.06, 14.86, 17.02, 18.65, 19.80, 20.47, 21.09, 21.78],
    },
}

## RCPs

**time periods:** 2000, 2005, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100<br>
**scenarios:** IMAGE RCP2.6, MiniCAM RCP4.5, AIM RCP6.0, MESSAGE RCP8.5<br>
**references:**<br>
**RCP3-PD (2.6) - IMAGE:** van Vuuren, D., M. den Elzen, P. Lucas, B. Eickhout, B. Strengers, B. van Ruijven, S. Wonink, R. van Houdt, 2007. Stabilizing greenhouse gas concentrations at low levels: an assessment of reduction strategies and costs. Climatic Change, doi: [10.1007/s/10584-006-9172-9](https://doi.org/10.1007/s/10584-006-9172-9);
RCP Database (Version 2.0.4) http://www.iiasa.ac.at/web-apps/tnt/RcpDb generated: 2010-04-15 15:22:57<br>
**RCP 4.5 - MiniCAM:** Clarke, L., J. Edmonds, H. Jacoby, H. Pitcher, J. Reilly, R. Richels, 2007. Scenarios of Greenhouse Gas Emissions and Atmospheric Concentrations. Sub-report 2.1A of Synthesis and Assessment Product 2.1 by the U.S. Climate Change Science Program and the Subcommittee on Global Change Research. Department of Energy, Office of Biological & Environmental Research, Washington, 7 DC., USA, 154 pp. https://www.pnnl.gov/main/publications/external/technical_reports/PNNL-16932.pdf
RCP Database (Version 2.0.4) http://www.iiasa.ac.at/web-apps/tnt/RcpDb; generated: 2010-04-15 15:22:57<br>
**RCP 6.0 - AIM:** Hijioka, Y., Y. Matsuoka, H. Nishimoto, M. Masui, and M. Kainuma, 2008. Global GHG emissions scenarios under GHG concentration stabilization targets. [Journal of Global Environmental Engineering](https://www.jsce.or.jp/publication/e/book/book_jgee.html#vol13) [13, 97-108](http://library.jsce.or.jp/jsce/open/00771/2008/13-0097.pdf);
RCP Database (Version 2.0.5) http://www.iiasa.ac.at/web-apps/tnt/RcpDb; generated: 2010-10-11 13:05:40<br>
**RCP 8.5 - MESSAGE:** Riahi, K., and Nakicenovic, N. (eds): 2007, [Greenhouse Gases - Integrated Assessment](https://www.sciencedirect.com/journal/technological-forecasting-and-social-change/vol/74/issue/7), Technological Forecasting and Social Change, Special Issue, 74(7), September 2007, 234 pp. https://doi.org/10.1016/j.techfore.2006.05.026 (ISSN 0040-1625);
RCP Database (Version 2.0.4) http://www.iiasa.ac.at/web-apps/tnt/RcpDb; generated: 2010-04-15 15:22:54<br>
[Scenario process for AR5 - CIESIN/Columbia U. IPCC DDC](https://web.archive.org/web/20250905023118/https://sedac.ciesin.columbia.edu/ddc/ar5_scenario_process/RCPs.html)<br>
[Scenarios for IPCC Fifth Assessment Report - IIASA](https://iiasa.ac.at/impacts/nov-2014/scenarios-for-ipcc-fifth-assessment-report)<br>
[Representative Concentration Pathways Database (RCP) - IIASA](https://iiasa.ac.at/models-tools-data/rcp)<br>
[RCP Database: Version 2.0.5 - IIASA](https://tntcat.iiasa.ac.at/RcpDb/dsd?Action=htmlpage&page=download)

In [ ]:
%%time
data = {
    "time": {"values": [1990, 2000, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100], "unit": "year"},
    "variable": "CO2 emissions - Total (World)",
    "unit": "PgC/yr",
    "scenarios": {
        "RCP3-PD (2.6) - IMAGE": [7.884, 9.167, 9.878, 10.260, 7.946, 5.024, 3.387, 2.035, 0.654, 0.117, -0.268, -0.420],
        "RCP 4.5 - MiniCAM": [7.884, 9.167, 9.518, 10.212, 11.170, 11.537, 11.280, 9.585, 7.222, 4.190, 4.220, 4.249],
        "RCP 6.0 - AIM": [7.884, 9.167, 9.389, 9.357, 9.438, 10.840, 12.580, 14.566, 16.477, 17.525, 14.556, 13.935],
        "RCP 8.5 - MESSAGE": [7.884, 9.167, 9.969, 12.444, 14.554, 17.432, 20.781, 24.097, 26.374, 27.715, 28.531, 28.817],
    },
}

## SSPs

**time periods:** 2005, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100<br>
**scenarios:** IAMC-IMAGE-ssp119, IAMC-IMAGE-ssp126, IAMC-MESSAGE-GLOBIOM-ssp245, IAMC-AIM-ssp370 (AIM/CGE SSP3-Baseline), IAMC-GCAM4-ssp434, IAMC-GCAM4-ssp460, IAMC-REMIND-MAGPIE-ssp534-over, IAMC-REMIND-MAGPIE-ssp585 (REMIND-MAGPIE SSP5-Baseline)<br>
**references:**<br>
[Shared Socioeconomic Pathways Scenario Database (SSP) - IIASA](https://iiasa.ac.at/models-tools-data/ssp)<br>
[Shared Socioeconomic Pathways: Scenario Explorer hosted by IIASA](https://ssp.apps.ece.iiasa.ac.at/iam-quantification-2018?type=table&variables=35&regions=1&runs=5,1,2,3,4,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127&dataHash=1c3294c996ea3b)

In [ ]:
%%time
data = {
    "time": {"values": [2005, 2010, 2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100], "unit": "year"},
    "variable": "CO2 emissions - Total (World)",
    "unit": "Gt CO2/yr",
    "scenarios": {
        "SSP1-19 - AIM/CGE": [34.3739, 35.7826, 37.2339, 19.0575, 8.2293, 1.7946, -2.1812, -3.7571, -4.3382, -4.3905, -4.4749],
        "SSP1-19 - GCAM4": [None, 35.7754, 37.6844, 19.3081, 10.5799, 3.2627, -0.6336, -4.3354, -8.421, -12.9638, -17.6742],
        "SSP1-19 - IMAGE": [33.1661, 35.4888, 33.2088, 17.968, 6.4139, 0.968, -1.65, -4.1369, -7.3702, -10.5677, -14.3404],
        "SSP1-19 - MESSAGE-GLOBIOM": [37.7208, 40.2656, 38.0939, 23.4042, 14.9339, 3.63, -4.0297, -8.1185, -11.9321, -14.3159, -15.4871],
        "SSP1-19 - REMIND-MAGPIE": [35.25, 36.25, 35.2, 26.31, 16.41, 3.761, -6.315, -9.558, -11.39, -12.77, -13.37],
        "SSP1-19 - WITCH-GLOBIOM": [31.922, 35.3034, 37.3123, 10.56, 7.4359, 3.0999, -0.6957, -3.7652, -7.2216, -9.2466, -9.6483],
        "SSP1-26 - AIM/CGE": [34.3739, 35.7818, 37.1813, 29.7905, 19.4314, 13.679, 8.6772, 4.1001, 2.0832, 0.627, 0.0177],
        "SSP1-26 - GCAM4": [None, 35.7754, 35.5266, 28.0269, 23.5731, 17.7668, 11.324, 4.921, -0.5556, -5.8802, -10.9639],
        "SSP1-26 - IMAGE": [33.1661, 35.4888, 38.3898, 33.9309, 25.9047, 17.7329, 10.5658, 4.7852, -2.9218, -8.0983, -8.3474],
        "SSP1-26 - MESSAGE-GLOBIOM": [37.7204, 40.2677, 38.4204, 32.7693, 28.1346, 19.7942, 9.6692, 1.6915, -4.1995, -9.0668, -12.2154],
        "SSP1-26 - REMIND-MAGPIE": [35.7798, 35.9142, 35.0616, 32.0562, 28.0504, 21.3754, 12.1056, 3.079, -2.7938, -6.7056, -8.588],
        "SSP1-26 - WITCH-GLOBIOM": [31.922, 35.3034, 39.6159, 17.3084, 15.0936, 12.7871, 10.4918, 7.2188, 4.3399, 0.3642, -4.939],
        "SSP1-34 - AIM/CGE": [34.3739, 35.7818, 37.1813, 35.2018, 31.2749, 26.476, 21.5056, 16.6316, 13.487, 10.8822, 9.0378],
        "SSP1-34 - GCAM4": [None, 35.7754, 35.5266, 34.6401, 33.5372, 30.0823, 27.3479, 20.8789, 11.0185, 4.1001, -1.3456],
        "SSP1-34 - IMAGE": [33.1661, 35.4888, 39.6388, 39.3514, 37.4431, 32.9473, 27.9556, 19.8133, 9.1234, -2.2754, -6.6155],
        "SSP1-34 - MESSAGE-GLOBIOM": [37.7205, 40.2677, 39.862, 38.4487, 35.7083, 32.9907, 27.037, 17.2956, 7.6326, -0.1029, -6.1741],
        "SSP1-34 - REMIND-MAGPIE": [35.7798, 35.9142, 34.9482, 37.1577, 37.438, 34.1812, 28.0783, 19.203, 9.8699, 2.4769, -2.8272],
        "SSP1-34 - WITCH-GLOBIOM": [31.922, 35.3034, 39.9652, 33.2526, 28.4091, 24.3865, 21.0793, 16.4961, 12.0979, 6.6008, -0.2423],
        "SSP1-45 - AIM/CGE": [34.3739, 35.7818, 36.5886, 39.4589, 38.4481, 35.0301, 31.9299, 28.427, 24.7902, 21.5447, 18.7381],
        "SSP1-45 - GCAM4": [None, 35.7754, 35.5266, 37.8637, 39.2495, 37.248, 35.7798, 32.4987, 24.5445, 17.4022, 15.3691],
        "SSP1-45 - IMAGE": [33.1661, 35.4888, 39.8601, 40.9581, 41.4986, 40.021, 37.7347, 32.323, 24.4556, 17.3077, 11.079],
        "SSP1-45 - MESSAGE-GLOBIOM": [37.7205, 40.2678, 40.5438, 41.9594, 41.4559, 41.2462, 40.1594, 35.7068, 28.3964, 18.6463, 8.9047],
        "SSP1-45 - REMIND-MAGPIE": [35.7798, 35.9142, 34.9693, 38.9416, 42.3109, 40.901, 36.4236, 30.2545, 23.3356, 16.5421, 12.1158],
        "SSP1-45 - WITCH-GLOBIOM": [31.922, 35.3034, 39.5882, 41.2656, 39.1855, 37.3502, 33.7811, 29.642, 25.7285, 18.1402, 9.6865],
        "SSP1-60 - WITCH-GLOBIOM": [31.922, 35.3034, 39.4591, 42.8967, 43.9289, 45.5231, 44.3028, 42.3248, 40.172, 37.6757, 31.675],
        "SSP1-Baseline - AIM/CGE": [34.3739, 35.8887, 39.1303, 41.8712, 42.1128, 40.1919, 38.8316, 35.9351, 33.4663, 31.6029, 29.8798],
        "SSP1-Baseline - GCAM4": [None, 35.7754, 36.8388, 41.7402, 46.5879, 48.0428, 49.1563, 48.0765, 44.0571, 39.6679, 34.8398],
        "SSP1-Baseline - IMAGE": [33.1661, 35.4888, 40.069, 42.6532, 43.7785, 42.4548, 41.6019, 39.2175, 33.3923, 28.6184, 24.6129],
        "SSP1-Baseline - MESSAGE-GLOBIOM": [37.7182, 40.2672, 41.2087, 43.5139, 45.6372, 47.2822, 49.5684, 50.0562, 49.2543, 49.5868, 49.2707],
        "SSP1-Baseline - REMIND-MAGPIE": [35.2499, 36.2509, 38.2048, 40.6691, 44.5641, 46.3531, 43.1878, 39.0578, 33.0532, 26.6236, 21.7717],
        "SSP1-Baseline - WITCH-GLOBIOM": [31.922, 35.3034, 41.3191, 44.2131, 48.1516, 47.2839, 46.603, 45.3935, 42.92, 39.3101, 31.2779],
        "SSP2-19 - AIM/CGE": [34.3739, 38.2054, 43.3381, 20.1386, 8.2232, 0.3992, -4.8961, -7.0386, -7.9892, -8.4073, -8.9869],
        "SSP2-19 - GCAM4": [None, 35.7754, 41.3377, 36.002, 9.5357, -7.4522, -13.1871, -14.1276, -18.178, -23.5224, -31.6837],
        "SSP2-19 - MESSAGE-GLOBIOM": [37.7672, 40.3135, 40.9313, 23.6331, 11.5241, 3.7789, -1.5107, -6.5404, -10.6089, -12.4109, -13.0489],
        "SSP2-19 - REMIND-MAGPIE": [35.09, 37.01, 42.31, 31.11, 8.623, -7.25, -10.48, -12.62, -13.64, -13.24, -14.18],
        "SSP2-26 - AIM/CGE": [34.3739, 38.2519, 43.2312, 30.4271, 18.7647, 10.5879, 6.5209, 1.0069, -1.5818, -3.6535, -4.8416],
        "SSP2-26 - GCAM4": [None, 35.7754, 39.8926, 46.3374, 24.6618, 4.0134, -4.4409, -6.5519, -11.1757, -17.079, -24.0182],
        "SSP2-26 - IMAGE": [33.1827, 35.6126, 39.7871, 28.6978, 18.55, 10.8425, 2.8712, -0.4699, -2.9248, -3.9852, -5.2415],
        "SSP2-26 - MESSAGE-GLOBIOM": [37.7708, 40.2943, 40.657, 34.5416, 26.3056, 16.3306, 9.2479, 2.0103, -4.0458, -6.4958, -8.3573],
        "SSP2-26 - REMIND-MAGPIE": [35.0755, 35.5959, 43.2856, 35.9917, 27.6483, 19.3515, 7.5494, -1.4445, -6.849, -10.4949, -12.0717],
        "SSP2-26 - WITCH-GLOBIOM": [31.922, 35.3034, 37.4229, 17.13, 15.2224, 13.3312, 10.1107, 6.9524, 3.1773, 0.6133, -3.7286],
        "SSP2-34 - AIM/CGE": [34.3739, 38.2519, 43.2312, 38.7326, 31.2436, 23.7069, 17.4882, 12.7307, 9.4446, 6.696, 4.2354],
        "SSP2-34 - GCAM4": [None, 35.7754, 39.8926, 46.3374, 39.8314, 25.4204, 13.7471, 7.4187, 2.7189, -3.4274, -10.4349],
        "SSP2-34 - IMAGE": [33.1827, 35.6145, 41.2109, 39.0743, 34.0681, 25.9235, 16.112, 13.3713, 7.1378, 2.881, 0.3072],
        "SSP2-34 - MESSAGE-GLOBIOM": [37.7708, 40.2943, 41.6859, 42.0316, 38.2197, 32.6119, 24.3027, 15.4234, 7.9744, 1.86, -1.4181],
        "SSP2-34 - REMIND-MAGPIE": [35.0755, 35.5959, 43.2691, 40.8878, 34.3304, 29.4215, 24.0969, 17.2968, 9.2118, 0.3554, -5.2282],
        "SSP2-34 - WITCH-GLOBIOM": [31.922, 35.3034, 39.7593, 29.683, 25.9122, 22.8045, 19.7284, 16.6409, 13.3991, 7.708, 2.4612],
        "SSP2-45 - AIM/CGE": [34.3739, 38.2519, 43.2312, 45.326, 46.2846, 40.6484, 34.091, 26.8606, 20.2881, 15.2502, 11.6656],
        "SSP2-45 - GCAM4": [None, 35.7754, 39.8926, 46.3374, 42.7904, 32.3017, 20.9787, 12.6946, 9.4622, 8.8659, 6.1285],
        "SSP2-45 - IMAGE": [33.1827, 35.6126, 42.6448, 46.7843, 45.4248, 42.1623, 36.3746, 31.4626, 23.5006, 16.0121, 14.9788],
        "SSP2-45 - MESSAGE-GLOBIOM": [37.7708, 40.2943, 42.2849, 44.3857, 44.4916, 42.8322, 39.0424, 34.1932, 25.8238, 15.4685, 9.115],
        "SSP2-45 - REMIND-MAGPIE": [35.0755, 35.5959, 43.266, 42.8401, 40.7524, 36.9163, 33.042, 29.4438, 26.2527, 22.4943, 21.9697],
        "SSP2-45 - WITCH-GLOBIOM": [31.922, 35.3034, 40.7819, 39.742, 39.4024, 34.5172, 30.9843, 27.9721, 24.9147, 20.7535, 13.737],
        "SSP2-60 - AIM/CGE": [34.3739, 38.2519, 43.2312, 46.6386, 49.2098, 48.5556, 47.5999, 46.5671, 45.9422, 45.5001, 44.9382],
        "SSP2-60 - GCAM4": [None, 35.7754, 39.8926, 46.3374, 50.6053, 51.9722, 51.8293, 50.2295, 45.5255, 38.5298, 28.7234],
        "SSP2-60 - IMAGE": [33.1827, 35.6126, 43.0631, 49.0232, 51.8339, 53.8201, 53.6763, 53.7241, 51.0194, 47.0254, 46.0603],
        "SSP2-60 - MESSAGE-GLOBIOM": [37.7708, 40.2943, 42.3922, 46.1252, 49.5448, 52.5857, 54.6635, 55.5372, 55.9741, 56.2188, 55.4263],
        "SSP2-60 - REMIND-MAGPIE": [35.0755, 35.5959, 43.2597, 44.9714, 49.0029, 51.8075, 51.9716, 51.1867, 49.3382, 44.7598, 41.9362],
        "SSP2-60 - WITCH-GLOBIOM": [31.922, 35.3034, 40.9937, 46.4843, 50.812, 50.8068, 50.5173, 50.1152, 47.4155, 44.1585, 36.8897],
        "SSP2-Baseline - AIM/CGE": [34.3739, 38.3755, 45.793, 51.7113, 56.6881, 59.8065, 61.8338, 63.6788, 65.6396, 68.3136, 70.7118],
        "SSP2-Baseline - GCAM4": [None, 35.7754, 41.4685, 49.5212, 56.867, 61.9235, 67.511, 71.2133, 72.6043, 73.7888, 75.3574],
        "SSP2-Baseline - IMAGE": [33.1827, 35.6126, 43.478, 49.4744, 52.9137, 55.9911, 57.6216, 60.8667, 64.4431, 67.8371, 72.4926],
        "SSP2-Baseline - MESSAGE-GLOBIOM": [37.769, 40.2946, 42.2624, 46.7276, 51.6065, 56.6516, 61.4023, 65.997, 74.1336, 81.0284, 85.6842],
        "SSP2-Baseline - REMIND-MAGPIE": [35.0905, 37.0096, 43.27, 48.6399, 56.5394, 60.4799, 67.4407, 70.4504, 72.54, 69.4599, 64.3597],
        "SSP2-Baseline - WITCH-GLOBIOM": [31.922, 35.3034, 43.952, 51.8725, 57.9626, 62.5435, 66.6327, 71.3462, 75.325, 77.9304, 73.3703],
        "SSP3-34 - AIM/CGE": [34.3739, 39.3986, 46.6377, 48.2922, 32.5528, 22.9309, 16.754, 9.4947, 3.9252, -0.7134, -3.8619],
        "SSP3-34 - IMAGE": [33.1779, 35.7104, 44.7148, 41.812, 29.4204, 19.4304, 8.2742, 4.2035, 3.3002, 3.1802, 2.8201],
        "SSP3-34 - MESSAGE-GLOBIOM": [37.7204, 40.3451, 47.8801, 47.0385, 31.2601, 21.8035, 12.8588, 6.1047, 2.2271, -0.1828, -0.7153],
        "SSP3-34 - WITCH-GLOBIOM": [31.922, 35.3034, 39.4283, 26.3234, 22.9757, 20.4554, 16.4079, 13.5896, 10.992, 5.9176, 1.7376],
        "SSP3-45 - AIM/CGE": [34.3739, 39.3986, 46.6377, 50.5891, 43.0262, 36.5619, 25.4704, 19.7139, 15.9589, 11.4, 6.4018],
        "SSP3-45 - IMAGE": [33.1779, 35.7104, 45.7028, 48.832, 43.697, 38.3183, 28.9404, 20.3699, 13.9083, 10.1226, 8.2186],
        "SSP3-45 - MESSAGE-GLOBIOM": [37.7168, 40.3744, 48.2529, 52.2746, 45.2494, 36.9094, 27.5329, 18.2116, 11.753, 7.2209, 4.7921],
        "SSP3-45 - WITCH-GLOBIOM": [31.922, 35.3034, 43.9966, 42.4446, 39.9711, 33.8283, 28.0426, 24.5205, 21.0812, 15.9874, 10.406],
        "SSP3-60 - AIM/CGE": [34.3739, 39.3986, 46.6377, 50.8053, 48.0745, 46.8881, 48.0484, 43.5267, 37.5542, 33.7122, 31.2482],
        "SSP3-60 - IMAGE": [33.1779, 35.7104, 46.5032, 53.1322, 54.2868, 56.0912, 52.6234, 46.02, 37.7846, 31.3229, 27.4345],
        "SSP3-60 - MESSAGE-GLOBIOM": [37.7205, 40.3727, 49.1496, 57.914, 60.315, 59.9811, 54.7114, 44.4708, 35.3239, 29.3898, 25.5462],
        "SSP3-60 - WITCH-GLOBIOM": [31.922, 35.3034, 45.4161, 52.2676, 55.1196, 52.5772, 47.6706, 44.3303, 39.0177, 31.9923, 26.9228],
        "SSP3-Baseline - AIM/CGE": [34.3739, 39.5608, 48.5027, 55.0009, 59.8773, 64.0014, 67.9623, 71.7927, 75.5715, 80.1372, 85.215],
        "SSP3-Baseline - GCAM4": [None, 35.7758, 44.6183, 53.6253, 60.2101, 64.8293, 70.1155, 74.7981, 78.2723, 80.967, 86.114],
        "SSP3-Baseline - IMAGE": [33.1779, 35.7104, 46.6496, 53.6693, 56.3442, 61.1271, 61.3071, 63.8387, 66.9267, 70.9804, 76.4771],
        "SSP3-Baseline - MESSAGE-GLOBIOM": [37.8062, 40.3811, 52.0613, 62.4478, 70.6995, 77.0191, 84.1347, 91.3238, 102.8427, 115.6809, 129.7206],
        "SSP3-Baseline - WITCH-GLOBIOM": [31.922, 35.3034, 48.287, 58.2936, 63.533, 66.7181, 68.3092, 72.4773, 75.9627, 80.3705, 84.1363],
        "SSP4-19 - WITCH-GLOBIOM": [31.922, 35.3034, 39.253, 12.7402, 9.286, 4.1926, -0.1842, -3.5386, -7.2457, -9.359, -10.8016],
        "SSP4-26 - AIM/CGE": [34.3739, 38.1982, 44.8181, 34.9675, 22.4191, 15.2162, 9.0045, 3.2987, 0.6722, -1.5518, -2.7853],
        "SSP4-26 - GCAM4": [None, 35.7758, 40.0488, 23.8218, 16.3242, 11.8814, 6.666, -0.1231, -10.623, -19.0914, -27.5445],
        "SSP4-26 - IMAGE": [33.1115, 35.6758, 37.4709, 27.0762, 16.5133, 6.0174, 1.7844, -0.0901, -1.035, -1.4936, -0.9609],
        "SSP4-26 - WITCH-GLOBIOM": [31.922, 35.3034, 41.9903, 22.1593, 17.7386, 13.8643, 9.2958, 6.6592, 3.8444, -0.465, -4.7809],
        "SSP4-34 - AIM/CGE": [34.3739, 38.1982, 44.8181, 41.4356, 35.7335, 29.219, 22.7722, 16.6672, 12.5219, 9.033, 6.3274],
        "SSP4-34 - GCAM4": [None, 35.7758, 40.0488, 35.0211, 28.0994, 19.9397, 14.7987, 10.2469, 2.7735, -7.132, -15.9621],
        "SSP4-34 - IMAGE": [33.1115, 35.6757, 41.0767, 39.3308, 34.6741, 24.8456, 15.321, 10.4495, 6.7794, 2.7016, 0.6638],
        "SSP4-34 - WITCH-GLOBIOM": [31.922, 35.3034, 41.9912, 36.8613, 30.6032, 24.3539, 20.4937, 16.2186, 11.8727, 5.598, 0.3949],
        "SSP4-45 - AIM/CGE": [34.3739, 38.1982, 44.8181, 46.2551, 46.0072, 40.8497, 35.463, 30.2517, 25.2502, 20.7386, 16.7182],
        "SSP4-45 - GCAM4": [None, 35.7758, 40.0488, 39.4511, 36.5661, 29.0346, 21.4804, 15.7154, 10.9315, 9.7826, 7.694],
        "SSP4-45 - IMAGE": [33.1923, 35.5837, 42.0861, 46.4136, 48.7907, 42.4438, 33.5338, 27.6916, 21.1188, 12.7455, 9.7988],
        "SSP4-45 - WITCH-GLOBIOM": [31.922, 35.3034, 41.6166, 43.8136, 41.1921, 36.8143, 32.8945, 29.6309, 23.9316, 19.3534, 11.5852],
        "SSP4-60 - GCAM4": [None, 35.7758, 40.0488, 46.1208, 48.9949, 49.0547, 47.6125, 44.591, 36.8058, 28.5648, 20.5693],
        "SSP4-60 - IMAGE": [33.1923, 35.5837, 42.1051, 46.7052, 49.7615, 49.2201, 47.7756, 47.5549, 47.0321, 42.7186, 41.1561],
        "SSP4-60 - WITCH-GLOBIOM": [31.922, 35.3034, 41.339, 45.971, 46.6743, 47.8875, 47.4644, 46.9065, 44.2359, 41.0056, 32.8489],
        "SSP4-Baseline - AIM/CGE": [34.3739, 38.2566, 45.1252, 49.7726, 50.5498, 47.2948, 43.7342, 40.5643, 38.9131, 37.8999, 36.7635],
        "SSP4-Baseline - GCAM4": [None, 35.7758, 41.636, 49.0696, 53.8469, 55.6644, 56.954, 55.8982, 51.1729, 47.5327, 44.7847],
        "SSP4-Baseline - IMAGE": [33.1818, 35.5945, 42.464, 48.2878, 50.8252, 50.3889, 48.4022, 47.7029, 47.0377, 44.7352, 44.3589],
        "SSP4-Baseline - WITCH-GLOBIOM": [31.922, 35.3034, 43.045, 47.2814, 50.5272, 49.7304, 48.9851, 48.9186, 47.2698, 42.4343, 33.654],
        "SSP5-19 - GCAM4": [None, 35.7754, 41.976, 35.6754, 4.9079, -8.9962, -9.6578, -9.4891, -12.1645, -18.1806, -26.3951],
        "SSP5-19 - REMIND-MAGPIE": [35.28, 36.37, 39.7, 37.77, 22.14, 1.254, -12.84, -16.25, -17.93, -19.55, -21.31],
        "SSP5-26 - AIM/CGE": [34.3739, 37.2297, 41.8081, 28.0629, 21.7403, 11.0077, 4.2979, -1.8447, -4.4437, -6.481, -7.6392],
        "SSP5-26 - GCAM4": [None, 35.7754, 39.4785, 51.1094, 23.6863, 0.1864, -3.1119, -3.7129, -6.7789, -12.7737, -21.666],
        "SSP5-26 - REMIND-MAGPIE": [35.2299, 36.335, 39.5613, 43.1646, 38.02, 24.9853, 7.4833, -3.9883, -11.3497, -16.4938, -18.9766],
        "SSP5-34 - AIM/CGE": [34.3739, 37.2297, 41.8081, 37.3896, 35.4775, 27.4798, 17.2479, 11.1832, 7.0819, 3.9777, 1.6142],
        "SSP5-34 - GCAM4": [None, 35.7754, 39.4785, 51.1094, 38.9118, 17.6772, 10.0899, 7.1783, 3.7726, -1.2527, -9.4378],
        "SSP5-34 - IMAGE": [33.1674, 35.9993, 44.0374, 45.2118, 40.0689, 29.5708, 16.8092, 9.4625, 6.5513, 2.4587, 0.7014],
        "SSP5-34 - REMIND-MAGPIE": [35.2299, 36.335, 39.5499, 44.474, 43.5223, 37.4209, 28.4542, 15.5094, 2.7482, -8.5353, -13.6001],
        "SSP5-34 - WITCH-GLOBIOM": [31.922, 35.3034, 37.8139, 18.9242, 19.5757, 18.5962, 18.1806, 17.8306, 17.6273, 15.58, 5.3557],
        "SSP5-45 - AIM/CGE": [34.3739, 37.2297, 41.8081, 43.3378, 44.6982, 36.6339, 27.3095, 22.0941, 20.1424, 15.4761, 11.8037],
        "SSP5-45 - GCAM4": [None, 35.7754, 39.4785, 51.1094, 48.4267, 32.7242, 21.9739, 15.8026, 11.4441, 13.6515, 14.2051],
        "SSP5-45 - IMAGE": [33.1674, 35.9993, 45.1235, 52.2617, 53.2681, 47.2155, 34.9504, 27.7631, 20.2589, 12.8879, 7.6007],
        "SSP5-45 - REMIND-MAGPIE": [35.2299, 36.335, 39.5425, 44.9341, 46.0024, 43.2244, 37.1714, 29.4774, 23.6499, 17.1663, 15.8607],
        "SSP5-45 - WITCH-GLOBIOM": [31.922, 35.3034, 39.8794, 32.1821, 32.6526, 32.2219, 31.3125, 29.7799, 27.3652, 24.8722, 14.496],
        "SSP5-60 - AIM/CGE": [34.3739, 37.2297, 41.8081, 44.7397, 48.3713, 45.7505, 41.9519, 40.1417, 39.2958, 38.4108, 37.2012],
        "SSP5-60 - GCAM4": [None, 35.7754, 39.4785, 51.1094, 59.1264, 58.7816, 57.4864, 52.7798, 40.9079, 30.9422, 21.7658],
        "SSP5-60 - IMAGE": [33.1682, 35.9825, 45.9513, 57.9475, 66.2722, 69.5887, 62.6355, 53.6939, 41.2712, 32.3227, 23.9956],
        "SSP5-60 - REMIND-MAGPIE": [35.2299, 36.335, 39.5604, 47.836, 54.5127, 58.173, 58.9398, 55.299, 50.5401, 44.1863, 40.9689],
        "SSP5-60 - WITCH-GLOBIOM": [31.922, 35.3034, 40.9541, 45.0476, 51.4851, 50.2298, 45.1993, 45.8068, 46.1934, 44.5405, 38.6313],
        "SSP5-Baseline - AIM/CGE": [34.3739, 37.4032, 44.275, 54.2289, 64.9723, 73.6589, 82.2667, 90.2872, 97.7439, 105.966, 114.441],
        "SSP5-Baseline - GCAM4": [None, 35.7754, 41.2035, 55.1423, 71.3284, 83.2515, 93.7571, 101.1123, 104.2832, 105.3475, 104.1186],
        "SSP5-Baseline - IMAGE": [33.1682, 35.9825, 46.6306, 60.2897, 72.7072, 86.4405, 94.7489, 105.5553, 110.4329, 112.4234, 111.91],
        "SSP5-Baseline - REMIND-MAGPIE": [35.2796, 36.3711, 44.6104, 56.7265, 69.8616, 84.4365, 101.3016, 117.4998, 129.4993, 130.3975, 126.0977],
        "SSP5-Baseline - WITCH-GLOBIOM": [31.922, 35.3034, 44.3215, 56.6257, 72.8812, 88.3262, 100.0683, 109.3883, 116.2267, 118.5059, 114.6223],
    },
}

## ScenarioMIP-CMIP7

**time periods:** 1750.5 ... 2300.5<br>
**scenarios:** high-overshoot, high-extension, verylow, verylow-overshoot, low, medium-extension, medium-overshoot, SSP2-COM<br>
High, Medium, Low, Very low, High-to-low, Medium-low, Low-to-negative<br>
**reference:**

In [ ]:
data = 0